# SoterAI v18 compact multilingual student

Upload `soterai-v18-colab-bundle.zip` to the root of your Google Drive, then choose **Runtime → Change runtime type → GPU**. Run the cells in order. The teacher is your existing v17 multilingual checkpoint. This trains a smaller candidate; improvement is not assumed. A new external release evaluation remains required.


In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, subprocess, sys
drive.mount('/content/drive')
BUNDLE = Path('/content/drive/MyDrive/soterai-v18-colab-bundle.zip')
WORK = Path('/content/soterai-v18').resolve()
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for member in archive.infolist():
        if not (WORK / member.filename).resolve().is_relative_to(WORK):
            raise ValueError('Unsafe archive path')
    archive.extractall(WORK)
subprocess.run([sys.executable, str(WORK/'scripts/ml/colab/run_v18_colab.py'), '--preflight'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK/'scripts/ml/colab/requirements-v18.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.__version__, torch.cuda.get_device_name(0))


## Train and verify ONNX

Checkpoints and candidate outputs persist on Drive. Default effective batch size is 64. If GPU memory is insufficient, reduce `BATCH_SIZE` to 8. Do not reduce the sequence length to hide memory problems. A 3-layer student may need more data or tuning; this run is an experiment.


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/soterai-v18-student-run')
EPOCHS, BATCH_SIZE = 5, 16
cmd = [sys.executable, str(WORK/'scripts/ml/colab/run_v18_colab.py'), '--output-dir', str(OUTPUT), '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE)]
# To resume checkpoint.pt in OUTPUT, append '--resume' to cmd.
subprocess.run(cmd, cwd=WORK, check=True)


## Inspect results and keep the candidate separate

Read the data report before interpreting scores. The teacher previously saw the original corpus, so the student internal test is not independent teacher evidence. Do not change the production model based on this notebook. Download the resulting directory and compare it on a fresh source/family-held-out release set, including per-language benign false alarms, tail attacks and CPU latency. The bundle includes `compare-v18-bases.py`; locally add `--candidate-dir models/ml-classifier-v18-student` to the documented paired comparison command.


In [ ]:
import json
for name in ['training_stats.json', 'eval_results.json']:
    p = OUTPUT/name
    if p.exists():
        result = json.loads(p.read_text())
        print(name, json.dumps(result, indent=2)[:16000])
print('Candidate files:', sorted(p.name for p in OUTPUT.iterdir()))
